# WESAD Stress Classifier

Trains a binary stress classifier on WESAD wrist data (HR + ACC). Run this once you have WESAD downloaded to `ml/data/WESAD/`.

In [ ]:
# Cell 1: Load + parse WESAD
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import find_peaks

DATA_DIR = Path('data/WESAD')
SUBJECTS = [f'S{i}' for i in range(2, 18) if i != 12]
WRIST_FS_BVP = 64
WRIST_FS_ACC = 32
LABEL_FS = 700
WIN_SEC = 60
STRIDE_SEC = 5

def load_subject(sid):
    with open(DATA_DIR / sid / f'{sid}.pkl', 'rb') as f:
        d = pickle.load(f, encoding='latin1')
    return {
        'bvp': d['signal']['wrist']['BVP'].squeeze(),
        'acc': d['signal']['wrist']['ACC'],
        'label': d['label'],
    }

In [ ]:
# Cell 2: Feature extraction
def bvp_to_hr(bvp, fs=WRIST_FS_BVP):
    peaks, _ = find_peaks(bvp, distance=int(fs*0.4), prominence=np.std(bvp)*0.5)
    if len(peaks) < 2: return np.array([])
    rr = np.diff(peaks) / fs
    hr = 60.0 / rr
    n_secs = int(len(bvp)/fs)
    if n_secs <= 0: return np.array([])
    return np.interp(np.arange(n_secs), peaks[:-1]/fs, hr)

def acc_magnitude(acc, fs=WRIST_FS_ACC):
    mag = np.linalg.norm(acc, axis=1)
    n_secs = int(len(mag)/fs)
    if n_secs <= 0: return np.array([])
    return np.array([mag[i*fs:(i+1)*fs].mean() for i in range(n_secs)])

def feature_window(hr_window, motion_window):
    if len(hr_window) < 5 or len(motion_window) < 5:
        return None
    return np.array([
        hr_window.mean(), hr_window.std(), hr_window.max() - hr_window.min(),
        np.polyfit(np.arange(len(hr_window)), hr_window, 1)[0],
        motion_window.mean(), motion_window.std(), motion_window.max(),
        (motion_window > motion_window.mean() + motion_window.std()).sum(),
    ])

def labels_per_second(label, fs=LABEL_FS, n_secs=None):
    if n_secs is None: n_secs = int(len(label)/fs)
    return np.array([
        np.bincount(label[i*fs:(i+1)*fs]).argmax() if i*fs < len(label) else 0
        for i in range(n_secs)
    ])

def windows_for_subject(sid):
    s = load_subject(sid)
    hr = bvp_to_hr(s['bvp'])
    mo = acc_magnitude(s['acc'])
    n = min(len(hr), len(mo))
    if n < WIN_SEC: return [], [], []
    lab1hz = labels_per_second(s['label'], n_secs=n)
    feats, labels, owners = [], [], []
    for start in range(0, n - WIN_SEC, STRIDE_SEC):
        f = feature_window(hr[start:start+WIN_SEC], mo[start:start+WIN_SEC])
        if f is None: continue
        win_label = lab1hz[start:start+WIN_SEC]
        majority = np.bincount(win_label).argmax()
        if majority not in (1, 2, 3, 4): continue
        feats.append(f)
        labels.append(1 if majority == 2 else 0)
        owners.append(sid)
    return feats, labels, owners

In [ ]:
# Cell 3: Train RandomForest with leave-one-subject-out CV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

X_all, y_all, owner_all = [], [], []
for sid in SUBJECTS:
    f, l, o = windows_for_subject(sid)
    X_all.extend(f); y_all.extend(l); owner_all.extend(o)
X_all = np.array(X_all); y_all = np.array(y_all); owner_all = np.array(owner_all)
print(f'Total windows: {len(X_all)} | stress: {y_all.sum()} | non-stress: {(y_all==0).sum()}')

per_subj_f1 = []
for held in SUBJECTS:
    train_mask = owner_all != held
    test_mask = owner_all == held
    if test_mask.sum() == 0 or y_all[test_mask].sum() == 0: continue
    scaler = StandardScaler().fit(X_all[train_mask])
    Xtr = scaler.transform(X_all[train_mask]); Xte = scaler.transform(X_all[test_mask])
    clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0).fit(Xtr, y_all[train_mask])
    pred = clf.predict(Xte)
    f1 = f1_score(y_all[test_mask], pred, zero_division=0)
    per_subj_f1.append(f1)
    print(f'{held}: F1 = {f1:.3f}')
print(f'\nMean LOSO F1: {np.mean(per_subj_f1):.3f}')

In [ ]:
# Cell 4: Train final MLP on all data + export TFLite
import tensorflow as tf
import json

scaler = StandardScaler().fit(X_all)
X_scaled = scaler.transform(X_all)

mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_scaled.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
mlp.fit(X_scaled, y_all, epochs=30, batch_size=64, validation_split=0.2, verbose=0)

converter = tf.lite.TFLiteConverter.from_keras_model(mlp)
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f: f.write(tflite_model)

with open('feature_scaler.json', 'w') as f:
    json.dump({'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}, f)

print(f'Saved model.tflite ({len(tflite_model)/1024:.1f} KB)')